# 04 - Deep Learning Forecasting (PyTorch)

Train and compare six sequence architectures.


## Why Sequence Models

**Definition**  
Sequence models learn temporal dependencies directly from ordered windows.

**Theory**  
RNN/LSTM/GRU capture recurrence; CNN-LSTM captures local motifs then sequence context; TCN captures long receptive fields with dilated convolutions.

**Mathematical Intuition**  
Given sequence `X_{t-k:t-1}`, model learns mapping `f(X)->y_t` via gradient descent.

**Financial Intuition**  
Useful when nonlinear memory patterns exceed tabular lag interaction capacity.

**Business Impact**  
Potentially better under nonlinear shocks but costlier to train and tune.

**Real-World Example**  
GRU may generalize better than LSTM under small data due fewer parameters.

**Visual Explanation**  
Code cells below generate real plots from Apple market data.

**Code Explanation**  
Each code block is annotated inline and uses shared production modules from `src/`.

**Interpretation of Results**  
After each output, interpret what signal quality, risk, and forecasting reliability imply.


In [ ]:

import sys
import os
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = (PROJECT_ROOT / '..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src.forecast_pipeline import ForecastingFramework
from src.data_loader import load_stock_data, split_data
from src.features import create_features
from src.evaluation import regression_metrics
from src.visualization import *

OUT = PROJECT_ROOT / 'outputs'
OUT.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = PROJECT_ROOT / 'config' / 'config.yaml'
FAST_NOTEBOOK_MODE = os.getenv('NOTEBOOK_FULL_RUN', '0') != '1'

def make_framework():
    framework = ForecastingFramework(str(CONFIG_PATH))
    if FAST_NOTEBOOK_MODE:
        framework.config['models']['automl']['lazypredict'] = False
        framework.config['models']['automl']['pycaret'] = False
        framework.config['models']['automl']['flaml'] = False
        framework.config['models']['deep_learning']['epochs'] = 8
        framework.config['models']['deep_learning']['early_stopping_patience'] = 3
        framework.config['weight_optimization']['methods'] = ['grid']
    return framework

framework = make_framework()


In [ ]:

framework = make_framework()
framework.load_data()

# Deep section is compute-heavy; default to 1-day horizon for detailed architecture comparison.
deep_out = framework.train_deep_models(horizon=1)
display(deep_out['leaderboard'])

deep_out['leaderboard'].to_csv(OUT / 'tables/04_deep_learning_h1.csv', index=False)



## Deep Model Strengths and Weaknesses

- **Vanilla LSTM**: stable baseline sequence learner; may miss hierarchical patterns.
- **Stacked LSTM**: richer temporal hierarchy; higher overfitting risk.
- **Bidirectional LSTM**: powerful context extraction; less realistic for strict causal inference (uses both directions during training windows).
- **GRU**: efficient recurrent architecture with lower parameter count.
- **CNN-LSTM**: captures local motifs + long memory.
- **TCN**: strong parallelism and receptive-field control via dilation.
